# Projeto Final — Predição de Readmissão Hospitalar em Pacientes Diabéticos

**Curso:** Introdução ao Python para Pesquisadores em Biomedicina
**Entrega final:** limpeza + engenharia de atributos + modelagem (ML) + interpretação

---

**Nome:** _(preencha)_

**Link do seu repositório:** _(preencha)_

---

## O problema

A base **Diabetes 130-US Hospitals (1999–2008)** reúne cerca de 100 mil
internações de pacientes diabéticos em 130 hospitais dos EUA, com mais de 50
variáveis. A pergunta de pesquisa é **quais fatores se associam à readmissão
precoce** (em até 30 dias após a alta), tratando a readmissão como desfecho a
ser modelado.

Você vai construir um estudo completo e **reprodutível**: entender e limpar os
dados, criar atributos, treinar e **comparar modelos**, e **interpretar** os
resultados como faria num trabalho de pesquisa — com atenção a rigor
metodológico, diálogo com a literatura, limitações e honestidade nos achados.
Ao longo do caminho, você vai usar IA — e vai documentar **como** usou.


## ⚠️ Uso consciente de IA (leia antes de começar)

Você **pode e deve** usar assistentes de IA (Claude, ChatGPT, Copilot etc.).
A questão não é *se*, mas *como*. As regras deste projeto:

1. **Regra de ouro:** você só entrega o que entende. Esteja pronto para explicar
   qualquer linha do seu código e qualquer decisão de análise, oralmente, sem a IA.
2. **Registre o uso** no arquivo `USO_DE_IA.md` (modelo já no repositório): o que
   você pediu, o que a IA respondeu, e o que você fez com isso (aceitou, mudou,
   rejeitou) e **como verificou**.
3. **Verifique sempre.** IA erra com confiança. Código: rode e teste. Fatos
   clínicos (ex.: agrupamento de códigos CID-9, faixa normal de HbA1c): confira em
   fonte confiável, não confie na memória do modelo.
4. **Ética em pesquisa e dados de saúde.** Esta base é pública e desidentificada.
   Como treino para a sua vida de pesquisador(a): dados de saúde identificáveis
   exigem aprovação de comitê de ética (CEP/CONEP) e cuidado com a LGPD —
   **nunca** os cole em ferramentas de IA externas.

> Ao longo do notebook você verá blocos **🔎 Checkpoint de IA** — são os pontos
> onde a maioria das pessoas recorre à IA. Pare, documente e verifique.


## Fluxo Git (clonar o curso → criar repo próprio → publicar)

1. **Clone o repositório do curso:**
   ```bash
   git clone https://github.com/CURSO/projeto-final-biomedicina.git
   cd projeto-final-biomedicina
   ```
2. **Crie um repositório próprio** na sua conta do GitHub (*Fork* com um clique,
   ou um repositório novo e vazio pelo botão *New*).
3. **Aponte o clone para o seu repositório** (troque `SEU-USUARIO`):
   ```bash
   git remote set-url origin https://github.com/SEU-USUARIO/projeto-final-biomedicina.git
   ```
4. **Trabalhe** e vá salvando o progresso com commits frequentes.
5. **Publique no seu repositório:**
   ```bash
   git add .
   git commit -m "Projeto final concluído"
   git push -u origin main
   ```
6. **Entregue** o link do seu repositório ao professor.


## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import altair as alt
from functools import reduce

# ML (scikit-learn)
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, f1_score)

alt.data_transformers.enable("default", max_rows=100000)
pd.set_option("display.max_columns", 60)
print("Ambiente pronto.")

## 1. Baixar os dados

A base vem com valores ausentes codificados como `?`. Carregamos já tratando isso.

- **Opção A — `kagglehub`:** `kagglehub.dataset_download("brandao/diabetes")`
- **Opção B — `ucimlrepo`:** `fetch_ucirepo(id=296)`
- **Opção C — CSV no repositório:** `data/diabetic_data.csv`


In [ ]:
# Escolha UMA opção e ajuste o caminho.
# import kagglehub
# caminho = kagglehub.dataset_download("brandao/diabetes")
# CAMINHO_CSV = f"{caminho}/diabetic_data.csv"

CAMINHO_CSV = "data/diabetic_data.csv"

df = pd.read_csv(CAMINHO_CSV, na_values="?")
print("Formato:", df.shape)
df.head()

### O alvo: `readmitted`

In [ ]:
# Três classes originais: '<30', '>30', 'NO'.
df["readmitted"].value_counts()

# TODO (decisão sua): manter 3 classes ou binarizar para
# "readmitido em <30 dias" vs "não"? Justifique no README.
# Ex.: df["alvo"] = (df["readmitted"] == "<30").astype(int)

## 2. Conhecer os dados

Antes de qualquer modelo, entenda o que você tem em mãos.


In [ ]:
# Nulos por coluna (top 15)
df.isna().sum().sort_values(ascending=False).head(15)

In [ ]:
# Colunas numéricas vs categóricas
numericas = df.select_dtypes(include="number").columns.tolist()
categoricas = df.select_dtypes(exclude="number").columns.tolist()
print(f"{len(numericas)} numéricas, {len(categoricas)} categóricas")
df.describe()

> **🔎 Checkpoint de IA.** As colunas `diag_1`, `diag_2`, `diag_3` são códigos
> **CID-9**. É tentador pedir à IA para agrupá-los em categorias clínicas
> (circulatório, respiratório, diabetes...). Se fizer isso: **confira o
> mapeamento contra a tabela oficial do CID-9** — modelos costumam inventar
> faixas. Registre o prompt e a verificação em `USO_DE_IA.md`.


## Parte A — Limpeza e engenharia de atributos (POO)

Organize o pré-processamento em uma **classe**. Abaixo há um esqueleto com alguns
métodos prontos como exemplo e outros como `TODO` para você implementar.


In [ ]:
class PreprocessadorDiabetes:
    """Encapsula os passos de limpeza e engenharia de atributos.

    Uso:
        prep = PreprocessadorDiabetes()
        df_limpo = prep.transformar(df)
    """

    # colunas sem valor preditivo ou com vazamento/ID
    COLUNAS_DESCARTAR = ["encounter_id", "patient_nbr", "weight",
                         "payer_code", "medical_specialty"]

    def __init__(self, limite_faltantes: float = 0.4):
        self.limite_faltantes = limite_faltantes
        self.relatorio_ = {}

    def _descartar_colunas(self, df: pd.DataFrame) -> pd.DataFrame:
        cols = [c for c in self.COLUNAS_DESCARTAR if c in df.columns]
        self.relatorio_["descartadas"] = cols
        return df.drop(columns=cols)

    def _remover_muito_faltante(self, df: pd.DataFrame) -> pd.DataFrame:
        frac = df.isna().mean()
        manter = frac[frac <= self.limite_faltantes].index
        return df[manter]

    def _codificar_idade(self, df: pd.DataFrame) -> pd.DataFrame:
        # 'age' vem como faixas: [0-10), [10-20)... Converte no ponto médio.
        if "age" in df.columns:
            meio = {f"[{i}-{i+10})": i + 5 for i in range(0, 100, 10)}
            df = df.assign(age_num=df["age"].map(meio))
        return df

    # TODO: implemente ao menos mais dois métodos, por exemplo:
    #   _agrupar_diagnosticos(self, df)  -> agrupa CID-9 (verifique o mapa!)
    #   _tratar_faltantes_restantes(self, df) -> imputação justificada
    #   _colapsar_categorias_raras(self, df) -> junta níveis raros em "outro"

    def transformar(self, df: pd.DataFrame) -> pd.DataFrame:
        return (df
                .pipe(self._descartar_colunas)
                .pipe(self._remover_muito_faltante)
                .pipe(self._codificar_idade)
                # .pipe(self._agrupar_diagnosticos)
                # .pipe(self._tratar_faltantes_restantes)
               )


prep = PreprocessadorDiabetes()
df_limpo = prep.transformar(df)
print("Depois da limpeza:", df_limpo.shape)
print("Colunas descartadas:", prep.relatorio_["descartadas"])
df_limpo.head()

## Parte B — Transformações em estilo funcional

Aqui a ideia é escrever **funções puras** (recebem um `DataFrame`, devolvem um
novo, sem efeitos colaterais) e **compô-las**. Compare com a abordagem em classe
acima e comente no `USO_DE_IA.md`/conclusões qual achou mais legível.


In [ ]:
def pipe(*funcs):
    """Compõe funções: pipe(f, g, h)(x) == h(g(f(x)))."""
    return lambda x: reduce(lambda acc, f: f(acc), funcs, x)


# --- funções puras de transformação (df -> df) ---
def normalizar_genero(df):
    return df.assign(gender=df["gender"].replace("Unknown/Invalid", np.nan))

def binarizar_alvo(df):
    return df.assign(alvo=(df["readmitted"] == "<30").astype(int))

def marcar_muitas_internacoes(df):
    # atributo derivado: paciente com muitas visitas prévias
    total = df["number_inpatient"] + df["number_outpatient"] + df["number_emergency"]
    return df.assign(visitas_previas=total)

# TODO: escreva pelo menos mais uma função pura sua.

preparar = pipe(
    normalizar_genero,
    binarizar_alvo,
    marcar_muitas_internacoes,
)

df_feat = preparar(df_limpo)
df_feat[["gender", "readmitted", "alvo", "visitas_previas"]].head()

> **🔎 Checkpoint de IA.** Se pedir à IA para gerar funções de *feature
> engineering*, teste cada uma isoladamente (uma função pura é fácil de testar!):
> passe um `DataFrame` pequeno e confira a saída. Não confie sem rodar.


### Distribuição do alvo

In [ ]:
base = df_feat.dropna(subset=["alvo"])
alt.Chart(base).mark_bar().encode(
    x=alt.X("alvo:N", title="Readmitido em <30 dias (1=sim)"),
    y=alt.Y("count()", title="Nº de internações"),
    color=alt.Color("alvo:N", legend=None),
).properties(title="Distribuição do alvo (atenção ao desbalanceamento)", width=300)

## Parte C — Modelagem e comparação (ML)

Monte um `X`/`y`, separe treino e teste, e **compare pelo menos 3 modelos**.
Como as classes são desbalanceadas, olhe além da acurácia (F1, ROC-AUC, matriz
de confusão).


In [ ]:
dados = df_feat.dropna(subset=["alvo"]).copy()

alvo = "alvo"
# TODO: selecione conscientemente as colunas de X (evite vazamento, ex.: não use
# 'readmitted'). Comece com um subconjunto e justifique.
num_feats = ["age_num", "time_in_hospital", "num_lab_procedures",
             "num_medications", "number_diagnoses", "visitas_previas"]
cat_feats = ["gender", "race", "A1Cresult", "insulin"]

num_feats = [c for c in num_feats if c in dados.columns]
cat_feats = [c for c in cat_feats if c in dados.columns]

X = dados[num_feats + cat_feats]
y = dados[alvo]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

pre = ColumnTransformer([
    ("num", StandardScaler(), num_feats),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_feats),
])
print("Treino:", X_train.shape, "| Teste:", X_test.shape)

In [ ]:
class AvaliadorModelo:
    """Treina um modelo dentro de um pipeline e reporta métricas."""

    def __init__(self, nome, modelo, preprocessador):
        self.nome = nome
        self.pipe = Pipeline([("pre", preprocessador), ("clf", modelo)])

    def treinar(self, X, y):
        self.pipe.fit(X, y)
        return self

    def avaliar(self, X, y):
        pred = self.pipe.predict(X)
        proba = self.pipe.predict_proba(X)[:, 1]
        return {
            "modelo": self.nome,
            "f1": round(f1_score(y, pred), 3),
            "roc_auc": round(roc_auc_score(y, proba), 3),
        }


modelos = {
    "Regressão Logística": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    # TODO: adicione/troque modelos e ajuste hiperparâmetros conscientemente.
}

resultados = [
    AvaliadorModelo(nome, modelo, pre).treinar(X_train, y_train).avaliar(X_test, y_test)
    for nome, modelo in modelos.items()
]
pd.DataFrame(resultados).sort_values("roc_auc", ascending=False)

> **🔎 Checkpoint de IA.** Ao ajustar hiperparâmetros ou tratar
> desbalanceamento (SMOTE, `class_weight`...), é comum pedir receitas à IA.
> **Entenda o que cada opção faz** antes de aplicar — e cheque se a métrica que
> você está otimizando responde à sua pergunta de pesquisa. Reporte-a com
> honestidade, sem "caçar" o número mais bonito.


## Parte D — Interpretação e visualização

Vá além do número: **quais atributos** mais pesam? O que isso diz clinicamente?


In [ ]:
# Exemplo: importância de atributos do melhor modelo baseado em árvores.
# TODO: adapte para o seu melhor modelo e faça o gráfico com Altair.
melhor = AvaliadorModelo("RF", modelos["Random Forest"], pre).treinar(X_train, y_train)
clf = melhor.pipe.named_steps["clf"]

nomes = (num_feats +
         list(melhor.pipe.named_steps["pre"]
              .named_transformers_["cat"].get_feature_names_out(cat_feats)))
imp = (pd.DataFrame({"atributo": nomes, "importancia": clf.feature_importances_})
       .sort_values("importancia", ascending=False).head(15))

alt.Chart(imp).mark_bar().encode(
    x=alt.X("importancia:Q", title="Importância"),
    y=alt.Y("atributo:N", sort="-x", title=None),
).properties(title="Atributos mais importantes (top 15)", width=500)

## Conclusões (discussão de pesquisa)

Escreva 1–2 parágrafos no tom de uma **seção de discussão de artigo**. Sugestões:
- Qual modelo escolheria e por quê? A métrica escolhida responde à pergunta de pesquisa?
- Que atributos mais se associam à readmissão? Os achados **dialogam com a literatura**?
- Quais as limitações (dados de 1999–2008, viés de seleção, desbalanceamento,
  dados faltantes, generalização para outros contextos)?
- O que um(a) **revisor(a) questionaria**? O que faltaria para virar uma
  publicação sólida (validação externa, reprodutibilidade)?

_(escreva aqui)_


## Reflexão sobre o uso de IA

Preencha a reflexão final no arquivo **`USO_DE_IA.md`**. Em resumo, responda lá:
onde a IA ajudou de verdade, onde ela te induziu ao erro (e como você percebeu),
e o que você faria diferente na próxima vez.


---
### Como finalizar
```bash
git add .
git commit -m "Projeto final concluído"
git push -u origin main
```
Assim seu trabalho ficará salvo no repositório definitivo. 
Em caso de dúvidas e eventuais problemas, sintam-se a vontade de
enviar mensagem para o email dos professores e monitores do curso.
Vale comentar que o repositório deve estar público para podermos acessar.

--- 

### Finalização do curso

Esperamos que  o curso tenha sido de bom proveito para todos, foi uma grande experiência sairmos da nossa zona de conforto e vislumbrá-los com um pouco do nosso tão rico universo. Este projeto final é mais uma evidência do que estão agora capacitados a fazer. Contamos com um feedback do curso para possíveis futuras iterações e que esta seja uma de muitas colaborações entre o IDOR e o IMPATech.